In [2]:
import json
import os 


with open('config.json', 'r') as f:
    data = json.load(f)
    print(data["publications"])

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "duguid2014coordination")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Duguid_2014_StagHunt2_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

../study


In [3]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="duguid2014coordination"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [4]:
df['dyad_temp1'] = df['pair'].str.slice(0,3)
df['dyad_temp2'] = df['pair'].str.slice(3,6)


In [5]:
df.rename(columns={"cage3": "ape",
    "age_cage3":"age",
    "cage4":"ape_2",
    "age_cage4":"age_2",
    "initiator_name":"ape_initiator",
    "follower_name":"ape_follower",
    "cage3_leavehare":"cage3_leave_hare",
    "cage4_leavehare":"cage4_leave_hare",
    "total_leavehare":"total_leave_hare",
    "one_leavehare":"one_leave_hare",
    "both_leavehare":"both_leave_hare",
    "group":"group_original"}, inplace=True)

df = df.assign(role='focal_participant_1')
df = df.assign(role_2='focal_participant_2')

In [6]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['dyad_temp1'].replace(x, y, inplace=True)
    df['dyad_temp2'].replace(x, y, inplace=True)
    df['ape'].replace(x, y, inplace=True)
    df['ape_2'].replace(x, y, inplace=True)
    df['ape_initiator'].replace(x, y, inplace=True)
    df['ape_follower'].replace(x, y, inplace=True)

df['dyad']=df.dyad_temp1.str.cat(df.dyad_temp2, sep='_')

In [7]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [8]:
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2",
                   'age':'age_in_years',
                   "age_2":"age_in_years_2"}, inplace=True)

In [9]:
duguid2014coordination_standardized=df[['study_id','participant', 'age_in_years', 'sex','role',  
        'participant_2', 'age_in_years_2', 'sex_2', 'role_2', 'species','dyad', 'session', 'trial','study','condition',  'group_original', 'pairing_cage3',
       'pairing_cage4', 'stag_success', 'cage3_leave_hare', 'cage4_leave_hare',
       'total_leave_hare', 'one_leave_hare', 'both_leave_hare',
       'time_initiator_wait', 'ape_initiator', 'ape_follower',
       'checkback_hare_cage3', 'checkback_hare_cage4',
       'communication_hare_cage3', 'communication_hare_cage4',
       'communication_stag_cage3', 'communication_stag_cage4',
       'communication_stag_initiator' ]]


In [10]:
comp_out_path_stand = os.path.join(out_pathway, 'duguid2014coordination_exp2_standardized.csv')
duguid2014coordination_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =duguid2014coordination_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
duguid2014coordination_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'duguid2014coordination_exp2_glossary.csv')
duguid2014coordination_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
